In [1]:
!python3 -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

Defaulting to user installation because normal site-packages is not writeable
  Using cached python_woc-0.4.1.tar.gz (483 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached chardet-5.2.0-py3-none-any.whl.metadata (3.4 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached python_lzf-0.2.6.tar.gz (9.8 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached tqdm-4.70.1-py3-none-any

  error: subprocess-exited-with-error
  
  × Building wheel for python-woc (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [30 lines of output]
      Compiling C:\Users\spark\AppData\Local\Temp\pip-install-uw0cubpy\python-woc_0bb533ba9d5840b083409e4e54175017\woc\local.pyx because it changed.
      Compiling C:\Users\spark\AppData\Local\Temp\pip-install-uw0cubpy\python-woc_0bb533ba9d5840b083409e4e54175017\woc\tch.pyx because it changed.
      [1/2] Cythonizing C:\Users\spark\AppData\Local\Temp\pip-install-uw0cubpy\python-woc_0bb533ba9d5840b083409e4e54175017\woc\local.pyx
      [2/2] Cythonizing C:\Users\spark\AppData\Local\Temp\pip-install-uw0cubpy\python-woc_0bb533ba9d5840b083409e4e54175017\woc\tch.pyx
      running build_ext
      building 'woc.local' extension
      error: Microsoft Visual C++ 14.0 or greater is required. Get it with "Microsoft C++ Build Tools": https://visualstudio.microsoft.com/visual-cpp-build-tools/
      Traceback (most recent call last):
     

In [ ]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = ['mmorise/World']
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

In [ ]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

In [ ]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

In [ ]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [ ]:
# check if it has the right content
dfinf.head(1)

In [ ]:
#mode a means append, so you have all your projects in the same file
yournetid='your_net_id_here'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github